In [ ]:
!pip install datasets transformers torch soundfile librosa jiwer tqdm pandas

In [ ]:
import torch
import numpy as np
import re
import json
import csv
import soundfile as sf
from datetime import datetime
from pathlib import Path
from datasets import Dataset
from transformers import pipeline
from jiwer import wer, cer
from tqdm.auto import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [ ]:

LIBRISPEECH_PATH = r"D:\LibriSpeech\test-clean"

NUM_SAMPLES = 1500
MODEL_NAME = "openai/whisper-base"

OUTPUT_JSON = "librispeech_local_1500.json"
OUTPUT_CSV = "librispeech_local_1500.csv"

CHECKPOINT_FILE = "checkpoint_local.json"
SAVE_EVERY = 100
RESUME = True


In [ ]:
def normalize_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def save_checkpoint(results, index):
    checkpoint = {
        'last_index': index,
        'total_processed': len(results),
        'timestamp': datetime.now().isoformat(),
        'results': results
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False)

def load_checkpoint():
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None



In [ ]:
libri_path = Path(LIBRISPEECH_PATH)

data = []

for speaker_dir in libri_path.glob("*"):
    if not speaker_dir.is_dir():
        continue
    
    for chapter_dir in speaker_dir.glob("*"):
        if not chapter_dir.is_dir():
            continue
        
        trans_file = chapter_dir / f"{speaker_dir.name}-{chapter_dir.name}.trans.txt"
        if not trans_file.exists():
            continue
        
        with open(trans_file, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                parts = line.strip().split(' ', 1)
                if len(parts) != 2:
                    continue
                
                file_id, text = parts
                audio_file = chapter_dir / f"{file_id}.flac"
                
                if audio_file.exists():
                    data.append({
                        'id': file_id,
                        'audio_path': str(audio_file),  
                        'text': text,
                        'speaker_id': int(speaker_dir.name),
                        'chapter_id': int(chapter_dir.name)
                    })
                
                if len(data) >= NUM_SAMPLES:
                    break
        
        if len(data) >= NUM_SAMPLES:
            break
    
    if len(data) >= NUM_SAMPLES:
        break

print(len(data))

# create dataset
dataset = Dataset.from_dict({
    'id': [d['id'] for d in data],
    'audio_path': [d['audio_path'] for d in data],
    'text': [d['text'] for d in data],
    'speaker_id': [d['speaker_id'] for d in data],
    'chapter_id': [d['chapter_id'] for d in data]
})

print(f"  num_samples: {len(dataset)}")
print(f"  column_names: {dataset.column_names}")

In [ ]:
example = dataset[0]
print(example)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

transcriber = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=0 if device == "cuda" else -1,
    chunk_length_s=30,
    generate_kwargs={"language": "english"}
)


In [ ]:
def transcribe_sample(sample):
    """
    load audio from file path and trasncribe
    """
    try:
        
        audio_path = sample['audio_path']
        
        # soundfile to read FLAC
        audio_array, sample_rate = sf.read(audio_path)
        
        if sample_rate != 16000:
            import librosa
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sample_rate,
                target_sr=16000
            )
            sample_rate = 16000
        
        # transribe
        result = transcriber({
            'array': audio_array,
            'sampling_rate': sample_rate
        })
        
        return result["text"].strip()
    
    except Exception as e:
        return ""


# test
test_result = transcribe_sample(dataset[0])
if test_result:
    print(f"  original text: {dataset[0]['text'][:80]}...")
    print(f"  test_result: {test_result[:80]}...")
else:
    print("failed to transcribe")

In [ ]:
#checkpoint
start_idx = 0
results = []

if RESUME:
    checkpoint = load_checkpoint()
    if checkpoint:
        results = checkpoint['results']
        start_idx = checkpoint['last_index'] + 1

success_count = sum(1 for r in results if r['success'])

for i in tqdm(range(start_idx, len(dataset)), 
              initial=start_idx,
              total=len(dataset),
              desc="transcribing"):
    
    sample = dataset[i]
    
    reference_original = sample['text']
    prediction_original = transcribe_sample(sample)
    
    success = bool(prediction_original)
    if success:
        success_count += 1
    
    result = {
        'index': i,
        'id': sample['id'],
        'reference_text': reference_original,
        'transcription': prediction_original,
        'success': success
    }
    results.append(result)
    
    if (i + 1) % SAVE_EVERY == 0:
        save_checkpoint(results, i)
        tqdm.write(f"({i + 1}/{len(dataset)})")

print(f"total_samples: {len(results)}")
print(f"transcribe_results: {success_count} ({success_count/len(results)*100:.1f}%)")

In [ ]:

references_norm = []
predictions_norm = []

for result in results:
    if result['success']:
        ref_norm = normalize_text(result['reference_text'])
        pred_norm = normalize_text(result['transcription'])
        references_norm.append(ref_norm)
        predictions_norm.append(pred_norm)

if len(predictions_norm) > 0:
    wer_score = wer(references_norm, predictions_norm)
    cer_score = cer(references_norm, predictions_norm)
    
    print(f"  WER: {wer_score:.4f} ({wer_score*100:.2f}%)")
    print(f"  CER: {cer_score:.4f} ({cer_score*100:.2f}%)")
    
    metrics = {
        'wer': wer_score,
        'cer': cer_score,
        'total_samples': len(results),
        'successful_samples': len(predictions_norm),
        'success_rate': len(predictions_norm) / len(results)
    }
else:
    metrics = None

In [ ]:
output_data = {
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'dataset': 'LibriSpeech (local)',
        'source_path': LIBRISPEECH_PATH,
        'model': MODEL_NAME,
        'total_samples': len(results),
        'successful_transcriptions': success_count,
        'metrics': metrics
    },
    'results': results
}

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)


In [ ]:
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['index', 'id', 'reference_text', 'transcription', 'success']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


In [ ]:
#for 3 examples
success_samples = [r for r in results if r['success']][:3]

for i, sample in enumerate(success_samples, 1):
    print(f"\nssample {i}:")
    print(f"  ID: {sample['id']}")
    print(f"  original_text: {sample['reference_text'][:60]}...")
    print(f"  transcription: {sample['transcription'][:60]}...")
    
    ref_norm = normalize_text(sample['reference_text'])
    pred_norm = normalize_text(sample['transcription'])
    
    if ref_norm == pred_norm:
        print(f" perfect match")
    else:
        sample_wer = wer(ref_norm, pred_norm)
        print(f"  WER: {sample_wer:.2%}")